# CNN-Transformer HOSE Training on Colab

This notebook clones the repo, loads the pre-sharded training data prepared for GitHub, trains the 3-class CNN-Transformer (`Avoid`/`Hold`/`Buy`) on GPU if available, evaluates the best checkpoint with batched test inference, exports weekly score-ranked top-20 picks, and plots test-period portfolio performance versus VNINDEX.

In [ ]:
!nvidia-smi || true

In [ ]:
%cd /content
!rm -rf trading__weekly_CNN_Transformer
!git clone -b experiment/predicted-buy-only https://github.com/votranhuonggiang/trading__weekly_CNN_Transformer.git
%cd trading__weekly_CNN_Transformer
!pip install -q -r requirements.txt
!git log --oneline -n 3
!grep -n "build_top_k_portfolio" src/predict.py


In [ ]:
import sys
for module_name in list(sys.modules):
    if module_name == 'src' or module_name.startswith('src.'):
        del sys.modules[module_name]

from src.colab_data import load_sharded_arrays
from src.train import TrainingConfig, train_model
from src.model import CNNTransformerClassifier
from src.predict import build_portfolio_performance, build_top_k_portfolio, build_weekly_prediction_table, export_prediction_outputs
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

arrays = load_sharded_arrays('data/colab')
metadata = pd.read_parquet('data/colab/model_dataset_metadata.parquet')
vnindex_comparison = pd.read_csv('outputs/tables/vnindex_label_comparison.csv')

expected_metadata_cols = {
    'rebalance_date',
    'next_rebalance_date',
    'ticker',
    'label',
    'label_name',
    'next_week_return',
    'return_rank_pct',
    'split',
}
expected_label_names = {'Avoid', 'Hold', 'Buy'}
actual_label_names = set(metadata['label_name'].dropna().unique())
if not expected_metadata_cols.issubset(metadata.columns) or actual_label_names != expected_label_names:
    raise RuntimeError(
        'This notebook follows architecture_buy_only.md and requires 3-class labels (Avoid/Hold/Buy). '
        'Regenerate and commit data/colab with 3-class labels, then rerun in Colab. '
        f'Current metadata columns: {metadata.columns.tolist()}; labels: {sorted(actual_label_names)}'
    )

class_names = ['Avoid', 'Hold', 'Buy']

metadata['rebalance_date'] = pd.to_datetime(metadata['rebalance_date']).dt.normalize()
metadata['next_rebalance_date'] = pd.to_datetime(metadata['next_rebalance_date']).dt.normalize()
vnindex_comparison['rebalance_date'] = pd.to_datetime(vnindex_comparison['rebalance_date']).dt.normalize()
vnindex_comparison['next_rebalance_date'] = pd.to_datetime(vnindex_comparison['next_rebalance_date']).dt.normalize()

print({k: v.shape for k, v in arrays.items() if hasattr(v, 'shape')})
print('metadata_rows:', len(metadata))
print('metadata_cols:', metadata.columns.tolist())
print('label_mode: strict_three_class_buy_only_v2')
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print('gpu_name:', torch.cuda.get_device_name(0))


In [ ]:
config = TrainingConfig(
    epochs=15,
    batch_size=512 if torch.cuda.is_available() else 128,
    learning_rate=1e-3,
    patience=4,
    checkpoint_dir='checkpoints'
)
model, metrics = train_model(arrays, config)
print('best_epoch:', metrics['best_epoch'])
print('accuracy:', metrics['accuracy'])
print('macro_f1:', metrics['macro_f1'])
print(metrics['report'])
print('checkpoint:', metrics['checkpoint_path'])

In [ ]:
history = metrics['history']
epochs = [row['epoch'] for row in history]
train_loss = [row['train_loss'] for row in history]
val_accuracy = [row['val_accuracy'] for row in history]
val_macro_f1 = [row['val_macro_f1'] for row in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs, train_loss, marker='o', color='#1f4e79')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(epochs, val_accuracy, marker='o', label='Val Accuracy', color='#117a65')
axes[1].plot(epochs, val_macro_f1, marker='o', label='Val Macro F1', color='#b03a2e')
axes[1].set_title('Validation Metrics')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
device = torch.device(config.device)
num_classes = int(arrays['y_train'].max()) + 1
class_names = ['Avoid', 'Hold', 'Buy']
best_model = CNNTransformerClassifier(num_features=arrays['X_train'].shape[-1], num_classes=num_classes).to(device)
best_model.load_state_dict(torch.load(metrics['checkpoint_path'], map_location=device))
best_model.eval()

test_batch_size = 1024 if torch.cuda.is_available() else 256
test_loader = DataLoader(
    TensorDataset(
        torch.tensor(arrays['X_test'], dtype=torch.float32),
        torch.tensor(arrays['y_test'], dtype=torch.long),
    ),
    batch_size=test_batch_size,
    shuffle=False,
    num_workers=0,
)

preds = []
probs_list = []
targets = []
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        logits = best_model(x_batch.to(device))
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        pred = np.argmax(probs, axis=1)
        probs_list.append(probs)
        preds.append(pred)
        targets.append(y_batch.numpy())

probs = np.concatenate(probs_list)
preds = np.concatenate(preds)
y_test = np.concatenate(targets)

print('test_accuracy:', accuracy_score(y_test, preds))
print('test_macro_f1:', f1_score(y_test, preds, average='macro'))
print('predicted_class_counts:', dict(zip(class_names, np.bincount(preds, minlength=num_classes).tolist())))
print(classification_report(y_test, preds, target_names=class_names, digits=4, zero_division=0))



In [ ]:
cm = confusion_matrix(y_test, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Test Confusion Matrix')
plt.tight_layout()
plt.show()



In [ ]:
avg_probs = probs.mean(axis=0)
bar_colors = ['#7f8c8d', '#117a65'] if len(class_names) == 2 else ['#b03a2e', '#7f8c8d', '#117a65']
plt.figure(figsize=(7, 4))
plt.bar(class_names, avg_probs, color=bar_colors[:len(class_names)])
plt.title('Average Predicted Probability on Test Set')
plt.ylabel('Probability')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()



In [ ]:
all_x = np.concatenate([arrays['X_train'], arrays['X_val'], arrays['X_test']], axis=0)
all_probs = []
all_loader = DataLoader(
    TensorDataset(torch.tensor(all_x, dtype=torch.float32)),
    batch_size=1024 if torch.cuda.is_available() else 256,
    shuffle=False,
    num_workers=0,
)

with torch.no_grad():
    for (x_batch,) in all_loader:
        logits = best_model(x_batch.to(device))
        all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())

all_probs = np.concatenate(all_probs)
prediction_table = build_weekly_prediction_table(metadata, all_probs)

TOP_K = 20
top_table = build_top_k_portfolio(prediction_table, top_k=TOP_K)
top_table['rebalance_date'] = pd.to_datetime(top_table['rebalance_date']).dt.normalize()
score_path, top_path = export_prediction_outputs(prediction_table, 'outputs/predictions', top_k=TOP_K, buy_only=False)

# keep/exit/enter diagnostics based on weekly target-set transitions
rebalance_dates = sorted(top_table['rebalance_date'].unique())
prev = set()
turnover_rows = []
for d in rebalance_dates:
    cur = set(top_table.loc[top_table['rebalance_date'] == d, 'ticker'].tolist())
    keep = prev & cur
    exit_ = prev - cur
    enter = cur - prev
    turnover_rows.append({
        'rebalance_date': d,
        'n_keep': len(keep),
        'n_exit': len(exit_),
        'n_enter': len(enter),
        'turnover_ratio_est': (len(exit_) + len(enter)) / float(TOP_K) if TOP_K > 0 else 0.0,
    })
    prev = cur
turnover_diag = pd.DataFrame(turnover_rows)

fee_rates = [0.0, 0.0015, 0.002, 0.0025, 0.0035]
summary_rows = []
perf_tables = []
for fee_rate in fee_rates:
    perf = build_portfolio_performance(top_table, vnindex_comparison=vnindex_comparison, fee_rate=fee_rate)
    perf['rebalance_date'] = pd.to_datetime(perf['rebalance_date']).dt.normalize()
    perf = perf.merge(turnover_diag, on='rebalance_date', how='left')
    perf['fee_rate'] = fee_rate
    perf['trading_cost_est'] = fee_rate * perf['turnover_ratio_est'].fillna(0.0)
    perf_tables.append(perf)

    test_perf_fee = perf[perf['rebalance_date'] >= pd.Timestamp('2024-01-01')].copy()
    if len(test_perf_fee) == 0:
        continue

    port_std = test_perf_fee['portfolio_simple_return'].std()
    port_sharpe = np.sqrt(52) * test_perf_fee['portfolio_simple_return'].mean() / port_std if port_std > 0 else np.nan

    vn_sharpe = np.nan
    if 'vnindex_simple_return' in test_perf_fee.columns and test_perf_fee['vnindex_simple_return'].notna().any():
        vn_std = test_perf_fee['vnindex_simple_return'].std()
        vn_sharpe = np.sqrt(52) * test_perf_fee['vnindex_simple_return'].mean() / vn_std if vn_std > 0 else np.nan

    summary_rows.append({
        'fee_rate': fee_rate,
        'weeks_test': int(len(test_perf_fee)),
        'portfolio_total_return_test': float((1.0 + test_perf_fee['portfolio_simple_return']).prod() - 1.0),
        'portfolio_sharpe_test': float(port_sharpe) if pd.notna(port_sharpe) else np.nan,
        'vnindex_sharpe_test': float(vn_sharpe) if pd.notna(vn_sharpe) else np.nan,
        'avg_turnover_ratio_test': float(test_perf_fee['turnover_ratio_est'].mean()),
        'avg_trading_cost_est_test': float(test_perf_fee['trading_cost_est'].mean()),
        'avg_n_keep_test': float(test_perf_fee['n_keep'].mean()),
        'avg_n_exit_test': float(test_perf_fee['n_exit'].mean()),
        'avg_n_enter_test': float(test_perf_fee['n_enter'].mean()),
    })

perf_all = pd.concat(perf_tables, ignore_index=True)
summary_df = pd.DataFrame(summary_rows).sort_values('fee_rate').reset_index(drop=True)

base_fee_rate = 0.002
perf_base = perf_all[perf_all['fee_rate'] == base_fee_rate].copy()
performance_csv = 'outputs/predictions/weekly_top_20_buy_only_performance.csv'
summary_csv = 'outputs/predictions/weekly_top_20_buy_only_cost_sensitivity.csv'
perf_base.to_csv(performance_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

test_perf = perf_base[perf_base['rebalance_date'] >= pd.Timestamp('2024-01-01')].copy()
test_perf['portfolio_cumulative_test'] = (1.0 + test_perf['portfolio_simple_return']).cumprod()
if 'vnindex_simple_return' in test_perf.columns and test_perf['vnindex_simple_return'].notna().any():
    test_perf['vnindex_cumulative_test'] = (1.0 + test_perf['vnindex_simple_return']).cumprod()

print('score_csv:', score_path)
print('top_csv:', top_path)
print('performance_csv:', performance_csv)
print('cost_sensitivity_csv:', summary_csv)
print('base_fee_rate:', base_fee_rate)
display(summary_df)
display(test_perf.head(10))


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

required_cols = {'rebalance_date', 'portfolio_cumulative_test'}
missing = [c for c in required_cols if c not in test_perf.columns]
if missing:
    raise ValueError(f"test_perf is missing required columns: {missing}. Available: {test_perf.columns.tolist()}")

plot_df = test_perf.copy()
plot_df['rebalance_date'] = pd.to_datetime(plot_df['rebalance_date'], errors='coerce')
plot_df = plot_df.dropna(subset=['rebalance_date', 'portfolio_cumulative_test']).sort_values('rebalance_date')

if plot_df.empty:
    raise ValueError('No rows to plot after cleaning test_perf. Check upstream filtering and cumulative calculations.')

fig, ax = plt.subplots(figsize=(15, 7))
ax.plot(
    plot_df['rebalance_date'],
    plot_df['portfolio_cumulative_test'],
    label='Top 20 Buy-Only Portfolio (Test)',
    color='#117a65',
    linewidth=2.2,
)

if 'vnindex_cumulative_test' in plot_df.columns:
    vn = plot_df.dropna(subset=['vnindex_cumulative_test'])
    if not vn.empty:
        ax.plot(
            vn['rebalance_date'],
            vn['vnindex_cumulative_test'],
            label='VNINDEX (Test)',
            color='#1f4e79',
            linewidth=2.0,
        )

ax.set_title('Buy-Only Top 20 Portfolio vs VNINDEX - Test Period Only')
ax.set_xlabel('Rebalance Date')
ax.set_ylabel('Cumulative Growth of 1.0')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
